In [ ]:
!pip install google-play-scraper pandas

In [ ]:
from google_play_scraper import search

results = search("Opay", lang="en", country="ng", n_hits=5)
for app in results:
    print(app["title"], "->", app["appId"])

In [ ]:
from google_play_scraper import app

info = app("team.opay.pay", lang="en", country="ng")
print(info["title"])
print(info["developer"])
print(info["ratings"])

In [ ]:
import pandas as pd
from google_play_scraper import reviews, Sort

result, _ = reviews(
    "team.opay.pay",
    lang="en",
    country="ng",
    sort=Sort.NEWEST,
    count=300,
)

df = pd.DataFrame(result)
df = df[["at", "score", "content"]]
df.columns = ["date", "rating", "review"]

print(len(df), "reviews collected")
df.head()

In [ ]:
df.to_csv("opay_reviews_test.csv", index=False)
print("Saved!")

df["rating"].value_counts().sort_index()

In [ ]:
complaints = df[df["rating"] <= 2]

for text in complaints["review"]:
    print("-", text)

In [ ]:
from google_play_scraper import reviews, Sort
import pandas as pd

result, token = reviews(
    "team.opay.pay",
    lang="en",
    country="ng",
    sort=Sort.NEWEST,
    count=3000,
)

df = pd.DataFrame(result)[["at", "score", "content"]]
df.columns = ["date", "rating", "review"]

print(len(df), "reviews")
print("Oldest:", df["date"].min())
print("Newest:", df["date"].max())

df.to_csv("opay_reviews_3000.csv", index=False)

In [ ]:
from google_play_scraper import reviews, Sort
import pandas as pd

frames = []
for stars in [1, 2]:
    result, _ = reviews(
        "team.opay.pay",
        lang="en",
        country="ng",
        sort=Sort.NEWEST,
        count=3000,
        filter_score_with=stars,
    )
    frames.append(pd.DataFrame(result)[["at", "score", "content"]])

low = pd.concat(frames)
low.columns = ["date", "rating", "review"]
low = low.sort_values("date", ascending=False)

print(len(low), "low-star reviews")
print("Oldest:", low["date"].min())
print("Newest:", low["date"].max())

low.to_csv("opay_low_star.csv", index=False)

In [ ]:
oldest = low.groupby("rating")["date"].min()
print(oldest)

cutoff = oldest.max()
low = low[low["date"] >= cutoff].copy()

low["month"] = low["date"].dt.to_period("M")
print(low.groupby("month").size())

In [ ]:
clean = low[["date", "rating", "review"]].copy()
clean["review"] = (
    clean["review"]
    .fillna("")
    .str.replace(r"[\r\n]+", " ", regex=True)
    .str.strip()
)
clean = clean[clean["review"] != ""]

clean.to_csv("opay_complaints_clean.csv", index=False, encoding="utf-8-sig")
print(len(clean), "rows saved")

In [ ]:
import os
print(os.getcwd())

In [ ]:
import os
import pandas as pd

os.makedirs(r"C:\Data", exist_ok=True)

df = pd.read_csv("opay_low_star.csv", parse_dates=["date"])

# keep only the period both 1-star and 2-star reviews cover
df = df[df["date"] >= "2025-12-01 19:12:13"].copy()

# tidy the review text
df["review"] = (
    df["review"]
    .fillna("")
    .str.replace(r"[\r\n]+", " ", regex=True)
    .str.strip()
)
df = df[df["review"] != ""]

df.to_csv(r"C:\Data\opay_complaints_clean.csv", index=False, encoding="utf-8-sig")
print(len(df), "rows saved to C:\\Data\\opay_complaints_clean.csv")